In [7]:
## load .Env

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
### Importing PyPDFLoader and Reading PDF

In [6]:
from langchain_community.document_loaders import PyPDFLoader
PDF_PATH = "Project_Report[1].pdf"

loader  = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(len(pages))

52


In [9]:
print(pages[19].page_content[:200])

12 Chapter 3.  Method 
 
 
 
 
 
 
Figure 3.3: Correlation values 
 
 
which will have a major impact on the model’s performance. This will reduce a lot of 
strain on the Machine Learning model during


In [10]:
## Converting Pdf into chunks

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap = 100,
    separators = ["\n\n" , "\n" , " " ,"." , ","]
)

chunks = splitter.split_documents(pages)
print(len(chunks))

154


In [13]:
## Storing these Chunks into Chroma DB and create embeddings of it 

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings  = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(vector_store._collection.count())


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3532.93it/s]


462


In [25]:
## Retrieve number of chunks related to passed query

In [26]:
retriever = vector_store.as_retriever(search_kwargs={"k" : 3})
test_query = "what is machine learning ? explain breif about this. "
result = retriever.invoke(test_query)

for i,doc in enumerate(result , 1):
    print(f"----chunks{i}")
    print(doc.page_content[:200])
    print()
    print()

----chunks1
explicitly programmed. Machine Learning is defined as the computer program learns 
from experience E with respect to some class of tasks T and performance  measure P 
when its performance at tasks in 


----chunks2
explicitly programmed. Machine Learning is defined as the computer program learns 
from experience E with respect to some class of tasks T and performance  measure P 
when its performance at tasks in 


----chunks3
explicitly programmed. Machine Learning is defined as the computer program learns 
from experience E with respect to some class of tasks T and performance  measure P 
when its performance at tasks in 




In [28]:
### Creating RAG pipeline

In [42]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

SYSTEM_PROMPT = """\
    You are expert who have knowledge about every single term and explain short definition about that term.
    if you don't about the term clearly say no.
    Provide short and meaningful definition.

    context : {context}
    
    """

def format_docs(docs):
    return "\n\n----\n\n".join(docs.page_content for docs in docs)

prompt = ChatPromptTemplate.from_messages(
  [  ("system" , SYSTEM_PROMPT),
    ("human" , "{question}")
  ])

llm = ChatGroq(
    model = "llama-3.1-8b-instant",
    temperature = 0.6,
)

chain =(
    {"context" : retriever | format_docs , "question" : RunnablePassthrough()}
    | prompt
    | llm
    |StrOutputParser()
)

print("RAG is enabled.")

     

RAG is enabled.


In [43]:
question = "Machine Learning."
print(f"Q : {question}\n")
print("A:" , chain.invoke(question))

Q : Machine Learning.

A: **Machine Learning (ML):**
A subfield of Artificial Intelligence (AI) that enables systems to learn from data, identify patterns, and make predictions or decisions without being explicitly programmed.
